# GDGoC NUST x CYE AI Challenge 2026: Solar plant fault detection
# Team Epoch Fail

**Task:** for each 10-second sensor reading (DC voltage, DC current, irradiance, panel temperature) predict one of `normal`, `shading`, `short_circuit`, `degradation`, `open_circuit`.
**Metric:** macro F1 (each of the 5 classes counts equally).

**Approach in one paragraph.** Train and test come from different days, so every score here uses *leave-one-day-out* cross-validation. Faults in train only appear at high sunlight on 4 days, so a model on raw irradiance/temperature can learn "high sun = fault". Instead we fit a *normal-plant reference* (expected current from irradiance, expected voltage from temperature and irradiance) and give the model the **residuals** (observed - expected). Faults then show up as clean voltage drops: open circuit about -132 V, short circuit about -34 V, degradation about -14 V. Model: ExtraTrees with balanced class weights.

## 1. Setup and data loading
Works on Colab (upload the 3 CSVs when asked, or put them in the working folder) and on Kaggle.

In [ ]:
import os, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

def find(name):
    for p in [name, f'/content/{name}', *glob.glob(f'/kaggle/input/*/{name}'), *glob.glob(f'**/{name}', recursive=True)]:
        if os.path.exists(p): return p
    return None

if find('train.csv') is None:                      # Colab: ask for the files
    from google.colab import files
    files.upload()                                  # select train.csv, test.csv, sample_submission.csv

train = pd.read_csv(find('train.csv'))
test  = pd.read_csv(find('test.csv'))
print(train.shape, test.shape)
train.head()

## 2.
Class balance (shading is common, the three faults are rare) and which days the faults appear on.

In [ ]:
print(train.label.value_counts())
print()
print(pd.crosstab(train.day, train.label))
print()
print(train.groupby('label').irradiance.describe().round(0)[['min','50%','max']])

## 3. Traps we checked
* Faults only occur at high sun (800-1000 W/m2) and on days 8, 9, 11, 12, so raw irradiance/temperature can act as a shortcut.
* `short_circuit` and `open_circuit` come from two days only, so leave-one-day-out folds for days 11/12 are noisy.
* `id` is shuffled and test has no `day`, so rolling/lag features are not possible.
* Test has 360 rows below 160 V (train has 244). Check their irradiance below.

In [ ]:
low = test[test.dc_voltage < 160]
print('test rows < 160 V:', len(low))
print(low[['dc_voltage','dc_current','irradiance','panel_temp']].describe().round(1).T[['min','50%','max']])
print()
print(test[['dc_voltage','dc_current','irradiance','panel_temp']].describe().round(1).T[['min','50%','max']])
print(train[['dc_voltage','dc_current','irradiance','panel_temp']].describe().round(1).T[['min','50%','max']])

## 4. Feature engineering: normal-plant reference and residuals
Fit two simple regressions on **normal** training rows only:
* current ~ irradiance
* voltage ~ temperature + log(irradiance)

Residual = observed - expected. Plus a few ratio features (`I/G`, `P/G`, `V/I`).

In [ ]:
def fit_ref(df):
    n = df[df.label == 'normal']
    cI = np.linalg.lstsq(np.c_[np.ones(len(n)), n.irradiance], n.dc_current, rcond=None)[0]
    cV = np.linalg.lstsq(np.c_[np.ones(len(n)), n.panel_temp, np.log(n.irradiance)], n.dc_voltage, rcond=None)[0]
    return cI, cV

def make_features(d, ref):
    cI, cV = ref
    V, I, G, T = d.dc_voltage, d.dc_current, d.irradiance, d.panel_temp
    f = pd.DataFrame(index=d.index)
    expV = cV[0] + cV[1]*T + cV[2]*np.log(G)
    f['rI']    = I - (cI[0] + cI[1]*G)       # current residual
    f['rV']    = V - expV                    # voltage residual
    f['rVrel'] = f.rV / expV                 # relative voltage residual
    f['I_G']   = I / G                       # current per sunlight
    f['P_G']   = V*I / G                     # power per sunlight
    f['V_I']   = V / (I + 0.1)               # apparent resistance
    f['P']     = V*I
    f['V'], f['I'], f['G'], f['T'] = V, I, G, T
    return f

SETS = {
    'raw+ratios':           ['V','I','G','T','P','P_G','I_G','V_I'],
    'resid_only (no G,T)':  ['rI','rV','rVrel','I_G','P_G','V_I'],
    'resid + G (chosen)':   ['rI','rV','rVrel','I_G','P_G','V_I','G'],
    'everything':           ['rI','rV','rVrel','I_G','P_G','V_I','P','V','I','G','T'],
}
ref = fit_ref(train)
F = make_features(train, ref)
y, groups = train.label, train.day
print('reference coefficients  I~G:', ref[0].round(4), ' V~T,logG:', ref[1].round(3))

In [ ]:
order = ['normal','shading','degradation','short_circuit','open_circuit']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hi = train.irradiance > 800
for c in order:
    ax[0].hist(F.rV[(y==c) & hi], bins=60, alpha=.6, label=c)
ax[0].set_title('Voltage residual at high sun (>800 W/m2)'); ax[0].set_xlabel('observed - expected voltage (V)'); ax[0].legend()
for c in order:
    s = train[y==c].sample(min(1500, (y==c).sum()), random_state=0)
    ax[1].scatter(s.irradiance, s.dc_current, s=3, alpha=.5, label=c)
ax[1].set_title('Current vs irradiance'); ax[1].set_xlabel('irradiance'); ax[1].set_ylabel('current (A)')
plt.tight_layout(); plt.show()

## 5. Validation: leave-one-day-out CV
Each day is predicted by a model that never saw that day. We report pooled out-of-fold macro F1.

In [ ]:
def make_model(seed=0):
    return ExtraTreesClassifier(300, class_weight='balanced', min_samples_leaf=3, n_jobs=-1, random_state=seed)

def cv_predict(cols, model_fn=make_model):
    X = F[cols]; oof = np.empty(len(train), dtype=object)
    for tr, te in LeaveOneGroupOut().split(X, y, groups):
        oof[te] = model_fn().fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te])
    return oof

labels = sorted(y.unique()); rows = []; oofs = {}
for name, cols in SETS.items():
    oofs[name] = cv_predict(cols)
    pc = f1_score(y, oofs[name], average=None, labels=labels)
    rows.append({'features': name, 'macro_F1': f1_score(y, oofs[name], average='macro'), **dict(zip(labels, pc))})
res = pd.DataFrame(rows).set_index('features').round(3)
res

Per-class report and confusion matrix for the chosen feature set.

In [ ]:
best = 'resid + G (chosen)'
print(classification_report(y, oofs[best], digits=3))
cm = confusion_matrix(y, oofs[best], labels=order)
ConfusionMatrixDisplay(cm, display_labels=order).plot(xticks_rotation=45, cmap='Blues', values_format='d')
plt.show()

# per-day check. Macro F1 is meaningless on days with no faults (one false alarm = a class with F1 0),
# so show accuracy and fault false alarms / misses instead.
faults = ['degradation','short_circuit','open_circuit']
for d in sorted(train.day.unique()):
    m = (groups == d).values; p = pd.Series(oofs[best][m]); t_ = y[m].reset_index(drop=True)
    print(f'day {d:2d}: n={m.sum():5d}  accuracy={(p==t_).mean():.3f}  true faults={t_.isin(faults).sum():4d}  predicted faults={p.isin(faults).sum():4d}')

## 6. A second model for comparison (random forest, same features)

In [ ]:
rf = lambda: RandomForestClassifier(300, class_weight='balanced_subsample', min_samples_leaf=3, n_jobs=-1, random_state=0)
oof_rf = cv_predict(SETS[best], rf)
print('RandomForest macro F1: %.4f' % f1_score(y, oof_rf, average='macro'))
print('ExtraTrees   macro F1: %.4f' % f1_score(y, oofs[best], average='macro'))

## 7. Fit on all training data and predict test

In [ ]:
cols = SETS[best]
Ft, Fs = make_features(train, ref)[cols], make_features(test, ref)[cols]
final = ExtraTreesClassifier(500, class_weight='balanced', min_samples_leaf=3, n_jobs=-1, random_state=0).fit(Ft, y)
sub = pd.DataFrame({'id': test.id, 'label': final.predict(Fs)})
sub.to_csv('submission.csv', index=False)
print(sub.label.value_counts())
sub.head()

Sanity check: predicted fault groups in test should have the same electrical signature as in train (voltage residual about -132 / -34 / -14 V).

In [ ]:
chk = Fs.copy(); chk['pred'] = sub.label.values
chk.groupby('pred')[['rV','rI','G']].agg(['mean','std']).round(1)

In [ ]:
# Colab: download the file. Elsewhere this is a no-op.
try:
    from google.colab import files; files.download('submission.csv')
except Exception: pass

## 8. Results summary
Leave-one-day-out macro F1 (ExtraTrees): raw+ratios ~0.81, residuals only ~0.90, **residuals + irradiance ~0.92** (open_circuit ~1.00, short_circuit ~0.99, normal ~0.95, shading ~0.93, degradation ~0.75).
